# HyP3 Job Monitor & Downloader
Run the cells in order. Keep re-running **Cell 2** every 15-20 minutes until you see `RUNNING : 0`.
Then run **Cell 3** to download.

In [ ]:
# Cell 1 — Run once to set up
import hyp3_sdk as sdk
from collections import Counter
from pathlib import Path

JOB_NAME   = 'MyMine_PSInSAR'        
DOWNLOAD_DIR = Path('../data/interferograms')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

hyp3 = sdk.HyP3()
print('Connected to HyP3 successfully')
print('Credits remaining:', hyp3.check_credits())

Connected to HyP3 successfully
Credits remaining: 6320


In [13]:
batch  = hyp3.find_jobs(name=JOB_NAME)
counts = Counter(j.status_code for j in batch)

print(f'Total jobs : {len(batch)}')
print('-' * 25)
for status, count in sorted(counts.items()):
    icon = {'SUCCEEDED':  '✅', 'RUNNING': '⚙️ ', 'PENDING': '⏳', 'FAILED': '❌'}.get(status, '  ')
    bar  = '█' * (count // 5)   # simple progress bar
    print(f'  {icon}  {status:<12} {count:>4}  {bar}')

print('-' * 25)
print('Credits remaining:', hyp3.check_credits())

running = counts.get('RUNNING', 0) + counts.get('PENDING', 0)
if running == 0:
    print()
    print('ALL JOBS FINISHED — run Cell 3 to download!')
else:
    print(f'\n{running} jobs still running. Check again in 15-20 minutes.')

Total jobs : 168
-------------------------
  ❌  FAILED        111  ██████████████████████
  ✅  SUCCEEDED      57  ███████████
-------------------------
Credits remaining: 6320

ALL JOBS FINISHED — run Cell 3 to download!


In [16]:
import time
from pathlib import Path

batch     = hyp3.find_jobs(name=JOB_NAME)
succeeded = batch.filter_jobs(succeeded=True, running=False, pending=False)
failed    = batch.filter_jobs(succeeded=False, running=False, pending=False)

print(f'Succeeded : {len(succeeded)}')
print(f'Failed    : {len(failed)}')
print(f'Downloading to: {DOWNLOAD_DIR.resolve()}')
print()

# Download one by one with retry — skips already completed files
MAX_RETRIES = 5
total   = len(list(succeeded))
done    = 0
skipped = 0
errors  = []

for job in succeeded:
    # Check if already fully downloaded (file exists and size > 0)
    already_done = False
    if hasattr(job, 'files') and job.files:
        for f in job.files:
            fname = DOWNLOAD_DIR / f['filename']
            expected_size = f.get('size', 0)
            if fname.exists() and fname.stat().st_size == expected_size:
                already_done = True

    if already_done:
        skipped += 1
        done += 1
        print(f'[{done}/{total}] Skipped (already downloaded): {fname.name}')
        continue

    # Try downloading with retries
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            job.download_files(location=DOWNLOAD_DIR)
            done += 1
            print(f'[{done}/{total}] Downloaded: {job.job_id}')
            break
        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = attempt * 10   # wait 10s, 20s, 30s... between retries
                print(f'[{done}/{total}] Attempt {attempt} failed, retrying in {wait}s... ({e})')
                time.sleep(wait)
            else:
                print(f'[{done}/{total}] FAILED after {MAX_RETRIES} attempts: {job.job_id}')
                errors.append(job.job_id)

print()
print(f'Done. {done - len(errors)} downloaded, {skipped} skipped, {len(errors)} failed.')

if errors:
    print(f'\nFailed job IDs (re-run this cell to retry):')
    for e in errors:
        print(f'  {e}')

Succeeded : 57
Failed    : 0

[1/57] Skipped (already downloaded): S1AA_20231207T002123_20231219T002123_VVP012_INT80_G_weF_8A25.zip
[2/57] Skipped (already downloaded): S1AA_20231125T002123_20231219T002123_VVP024_INT80_G_weF_D952.zip
[3/57] Skipped (already downloaded): S1AA_20231125T002123_20231207T002123_VVP012_INT80_G_weF_5DC2.zip
[4/57] Skipped (already downloaded): S1AA_20231118T002930_20231130T002929_VVP012_INT80_G_weF_4989.zip


S1AA_20231113T002124_20231125T002123_VVP012_INT80_G_weF_0C76.zip: 100%|██████████| 283M/283M [00:11<00:00, 24.9MB/s] 


[5/57] Downloaded: 8e68fb38-11b2-4c06-b693-e203dc7fb19c


S1AA_20231106T002930_20231118T002930_VVP012_INT80_G_weF_EE43.zip: 100%|██████████| 285M/285M [00:11<00:00, 25.2MB/s] 


[6/57] Downloaded: 084ea154-5f81-4558-afb7-74ea2a246009


...

[55/57] Downloaded: c0c7f072-8619-4e6a-aec4-f12e20eedede


S1AA_20230110T002923_20230122T002922_VVP012_INT80_G_weF_D35F.zip: 100%|██████████| 285M/285M [00:11<00:00, 25.3MB/s] 


[56/57] Downloaded: 075a7a6a-4b0b-4d18-b696-51525c52c8b8


S1AA_20230105T002117_20230117T002116_VVP012_INT80_G_weF_5116.zip: 100%|██████████| 282M/282M [00:12<00:00, 24.3MB/s] 

[57/57] Downloaded: 09cb0980-d8fe-4365-a943-956fe1690c9d

Done. 57 downloaded, 4 skipped, 0 failed.
